In [ ]:
import pandas as pd

patients = pd.read_csv(
    "../data/processed/patients_cleaned.csv",
    parse_dates=["BIRTHDATE", "DEATHDATE"]
)

encounters = pd.read_csv("../data/raw/encounters.csv")

encounters.info()

# one to many relationship
encounters["PATIENT"].nunique()

# Count encounters per patient
encounters_per_patient = (encounters
                         .groupby("PATIENT")
                         .size()
                         .sort_values(ascending=False)
                         )
encounters_per_patient.head()


In [ ]:
# transform dates into datetimes

encounters["START"] = pd.to_datetime(encounters["START"], errors="coerce")
encounters["STOP"] =pd.to_datetime(encounters["STOP"], errors="coerce") 

# Calculate encounter duration (length of stay)
encounters["duration_hours"] = (encounters["STOP"] - encounters["START"]).dt.total_seconds() / 3600
encounters["duration_hours"].describe()


In [3]:
# Merge patients and encounters
merged = encounters.merge(
    patients[["Id", "FIRST", "LAST", "GENDER", "age", "age_group"]],
    left_on="PATIENT",
    right_on="Id",
    how="left"
)
merged.head()

,Id_x,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,...,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION,duration_hours,Id_y,FIRST,LAST,GENDER,age,age_group
0,f9ba83f7-9940-16ae-5f88-aea1af242f71,1973-02-01 07:34:23+00:00,1973-02-01 07:49:23+00:00,f9ba83f7-9940-16ae-0854-24bd34bf1843,497f39dd-280e-3d58-af5b-c5e3a3a09b10,2352c51f-5dce-383b-adbf-bf5cc1f1c4d7,e03e23c9-4df1-3eb6-a62d-f70f02301496,ambulatory,185347001,Encounter for problem (procedure),...,0.00,256355007.0,Glycine max (substance),0.250000,f9ba83f7-9940-16ae-0854-24bd34bf1843,Ashlee14,Tromp100,F,54.7,middleage
1,a1733070-046a-4506-13b1-f518d757cdd0,1997-04-22 16:12:20+00:00,1997-04-22 16:59:47+00:00,a1733070-046a-4506-bba6-47f32652e9d7,c64c848c-0315-3115-87fa-80c360bb6a73,35b13cfd-13a0-34e4-a83d-32af4b12344a,df166300-5a78-3502-a46a-832842197811,wellness,162673000,General examination of patient (procedure),...,554.20,NaN,NaN,0.790833,a1733070-046a-4506-bba6-47f32652e9d7,Donte636,Daugherty69,M,47.5,middleage
2,f9ba83f7-9940-16ae-81d1-8cdb486ca3ad,1973-02-16 21:34:23+00:00,1973-02-16 21:49:23+00:00,f9ba83f7-9940-16ae-0854-24bd34bf1843,497f39dd-280e-3d58-af5b-c5e3a3a09b10,2352c51f-5dce-383b-adbf-bf5cc1f1c4d7,e03e23c9-4df1-3eb6-a62d-f70f02301496,ambulatory,185347001,Encounter for problem (procedure),...,0.00,609328004.0,Allergic disposition (finding),0.250000,f9ba83f7-9940-16ae-0854-24bd34bf1843,Ashlee14,Tromp100,F,54.7,middleage
3,f9ba83f7-9940-16ae-158c-5cd0e5a140b9,1989-01-10 22:34:23+00:00,1989-01-10 22:49:23+00:00,f9ba83f7-9940-16ae-0854-24bd34bf1843,f22bc2ac-f8bd-3f9f-9a40-93db37639d3f,b13a2be9-5dba-3041-b1c6-62ff62c512ea,e03e23c9-4df1-3eb6-a62d-f70f02301496,wellness,410620009,Well child visit (procedure),...,0.00,NaN,NaN,0.250000,f9ba83f7-9940-16ae-0854-24bd34bf1843,Ashlee14,Tromp100,F,54.7,middleage
4,a1733070-046a-4506-0d73-cf2822965f28,2013-05-14 16:12:20+00:00,2013-05-14 17:09:30+00:00,a1733070-046a-4506-bba6-47f32652e9d7,c64c848c-0315-3115-87fa-80c360bb6a73,35b13cfd-13a0-34e4-a83d-32af4b12344a,df166300-5a78-3502-a46a-832842197811,wellness,162673000,General examination of patient (procedure),...,1616.16,NaN,NaN,0.952778,a1733070-046a-4506-bba6-47f32652e9d7,Donte636,Daugherty69,M,47.5,middleage


In [4]:
# encounter types
encounters["ENCOUNTERCLASS"].value_counts()

ENCOUNTERCLASS
ambulatory    2781
wellness      1344
outpatient     745
urgentcare     270
emergency      191
home            62
inpatient       55
snf             12
hospice          9
virtual          4
Name: count, dtype: int64